In [0]:
df = spark.table("bankingdata.bronze.transactions")
display(df.limit(10))

In [0]:
df.columns

### Cast body → string

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("body_string", col("body").cast("string"))

In [0]:
display(df.limit(10))

### Define schema

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("account_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("branch_id", IntegerType(), True),
    StructField("card_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("location", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("is_high_value", BooleanType(), True),
    StructField("_corrupt_record", StringType(), True)
])

### Parse JSON

In [0]:
#df = df.withColumn("json_data", from_json(col("body_string"), schema))

In [0]:
df = df \
    .withColumn(
        "json_data",
        from_json(
            col("body_string"),
            schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record"
            }
        )
    )

In [0]:
display(df.limit(10))

### Flatten

In [0]:
#df = df.select("json_data.*", "enqueuedTime")

In [0]:
df = df.select(
    "json_data.*",
    "enqueuedTime"
)

In [0]:
display(df.limit(10))


### Primary Key Validation

In [0]:
df = df.filter(
    # transaction_id validation
    col("transaction_id").isNotNull() &
    (trim(col("transaction_id")) != "") &
    (col("transaction_id") != "0") &

    # account_id validation
    col("account_id").isNotNull() &
    (trim(col("account_id")) != "") &
    (col("account_id") != "0") &

    # amount validation
    col("amount").isNotNull() &
    (col("amount") > 0)
)

### Handling Corrupted Records

In [0]:
pk_col = "transaction_id" 

df_bad = df.filter(col("_corrupt_record").isNotNull()) \
    .select(
        col(pk_col),
        col("_corrupt_record"),
        current_timestamp().alias("error_time")
    )

df_bad.write.format("delta").mode("append").saveAsTable("bankingdata.silver.trancation_corruptedtable")

In [0]:
%sql
select * from bankingdata.bronze.trancation_corruptedtable limit 10

In [0]:
df = df.filter(col("_corrupt_record").isNull()).drop("_corrupt_record")

### Business Quality Check

In [0]:
df.columns

In [0]:
from pyspark.sql.functions import col, when, to_date, current_timestamp

df = (
    df
    .withColumn(
        "transaction_direction",
        when(col("transaction_type").isin("withdrawal", "purchase", "transfer_out"), "DEBIT")
        .when(col("transaction_type").isin("deposit", "refund", "transfer_in"), "CREDIT")
        .otherwise("UNKNOWN")
    )
    .withColumn(
        "transaction_category",
        when(col("amount") >= 10000, "HIGH_VALUE")
        .when((col("amount") >= 1000) & (col("amount") < 10000), "MEDIUM_VALUE")
        .otherwise("LOW_VALUE")
    )
    .withColumn(
        "is_high_value",
        when(col("amount") >= 10000, True).otherwise(False)
    )
    .withColumn(
        "processing_delay_seconds",
        col("enqueuedTime").cast("long") - col("event_time").cast("long")
    )
    .withColumn(
        "event_date",
        to_date(col("event_time"))
    )
    .withColumn(
        "silver_created_at",
        current_timestamp()
    )
)

### Removing Duplicates

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("transaction_id").orderBy(col("enqueuedTime").desc())

df = (
    df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
# WRITE TO UNITY CATALOG
silver_table = "bankingdata.silver.transaction"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)